# HTL Paper — ERCO Sample: GAM + XGB Stacked Load Forecast
- **Load data**: EIA 930 via PUDL (filtered to ERCO)
- **Temperature**: `ERCO_T2_wy2022_hourly.csv` from CONUS404 extraction
- **Split**: 75% train (Oct 2021 – Jun 2022) / 25% test (Jul – Sep 2022)
- **Model**: LinearGAM + XGBoost residual stacking with sliding-window lambda update

In [ ]:
import os
import numpy as np
import pandas as pd
import holidays
import xgboost as xgb
import matplotlib.pyplot as plt
from pygam import LinearGAM, s, f, te
from sklearn.model_selection import GridSearchCV
from scipy.optimize import minimize_scalar
from tqdm import tqdm

print(os.uname().nodename)

## 1. Load EIA 930 via PUDL — ERCO only

In [ ]:
# Load full EIA 930 from PUDL and immediately filter to ERCO
# Filtering early keeps memory manageable
df_y = pd.read_parquet(
    "s3://pudl.catalyst.coop/nightly/core_eia930__hourly_operations.parquet",
    dtype_backend="pyarrow",
)

df_y = df_y[df_y["balancing_authority_code_eia"] == "ERCO"].copy()
df_y["datetime_utc"] = pd.to_datetime(df_y["datetime_utc"], utc=True)
df_y = df_y.sort_values("datetime_utc").reset_index(drop=True)

print(f"ERCO rows: {len(df_y):,}")
print(f"Date range: {df_y['datetime_utc'].min()} → {df_y['datetime_utc'].max()}")
print(df_y.columns.tolist())

## 2. Feature Engineering — Lags, Calendar, Holidays

In [ ]:
# Lagged load
df_y["load_24"]  = df_y["demand_adjusted_mwh"].shift(24)
df_y["load_168"] = df_y["demand_adjusted_mwh"].shift(168)

# Calendar features
us_holidays = holidays.US()
df_y["hour"]       = df_y["datetime_utc"].dt.hour
df_y["dayofweek"]  = df_y["datetime_utc"].dt.dayofweek
df_y["dayofyear"]  = df_y["datetime_utc"].dt.dayofyear
df_y["is_holiday"] = df_y["datetime_utc"].dt.date.apply(
    lambda x: int(x in us_holidays)
)

print(df_y[["datetime_utc", "demand_adjusted_mwh", "load_24", "load_168",
            "hour", "dayofweek", "dayofyear", "is_holiday"]].head())

## 3. Load ERCO Temperature from CONUS404 (WY2022)

In [ ]:
# Load the ERCO temperature CSV produced by the CONUS404 extraction notebook
df_temp = pd.read_csv("ERCO_T2_wy2022_hourly.csv")
df_temp["time"] = pd.to_datetime(df_temp["time"], utc=True)
df_temp = df_temp.rename(columns={"time": "datetime_utc", "T2_K": "avg_temp_K"})
df_temp = df_temp[["datetime_utc", "avg_temp_K"]]

print(f"Temp rows: {len(df_temp):,}")
print(f"Date range: {df_temp['datetime_utc'].min()} → {df_temp['datetime_utc'].max()}")
df_temp.head()

## 4. Smoothed Temperature Features

In [ ]:
# Exponential smoothing — captures heat accumulation and prolonged heat stress
# alpha_95: slow decay; alpha_99: very slow decay (following base paper)
df_temp = df_temp.sort_values("datetime_utc").reset_index(drop=True)

df_temp["temp_s95"] = df_temp["avg_temp_K"].ewm(alpha=0.05, adjust=False).mean()
df_temp["temp_s99"] = df_temp["avg_temp_K"].ewm(alpha=0.01, adjust=False).mean()

print(df_temp[["datetime_utc", "avg_temp_K", "temp_s95", "temp_s99"]].head(10))

## 5. Merge Load + Temperature

In [ ]:
# Merge on datetime — inner join keeps only hours present in both
df = df_y.merge(df_temp, on="datetime_utc", how="inner")

# Drop rows with NaNs in required columns
required = ["demand_adjusted_mwh", "load_24", "load_168", "avg_temp_K"]
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=required)
df = df.sort_values("datetime_utc").reset_index(drop=True)

print(f"Merged rows: {len(df):,}")
print(f"Date range: {df['datetime_utc'].min()} → {df['datetime_utc'].max()}")
df.head()

## 6. Train / Test Split — 75% / 25%
- **Train**: Oct 2021 – Jun 2022 (~9 months, includes full cool season + spring)
- **Test**: Jul – Sep 2022 (~3 months, summer heat peak — most relevant for HTL)


In [ ]:
# 75/25 chronological split
n_total = len(df)
n_train = int(n_total * 0.75)

df_train = df.iloc[:n_train].copy()
df_test  = df.iloc[n_train:].copy()

FEATURES = ["hour", "dayofweek", "dayofyear", "is_holiday",
            "load_24", "load_168", "avg_temp_K", "temp_s95", "temp_s99"]
TARGET   = "demand_adjusted_mwh"

for name, split in [("train", df_train), ("test", df_test)]:
    print(f"{name}: {len(split):,} rows  "
          f"({split['datetime_utc'].min().date()} → {split['datetime_utc'].max().date()})")

print(f"\nCutoff date: {df_train['datetime_utc'].max().date()}")

## 7. Fit GAM on Training Data

In [ ]:
y_train = df_train[TARGET].values
X_train = df_train[FEATURES].to_numpy(dtype=float)

gam = LinearGAM(
    s(0, n_splines=24, basis="cp") +   # hour (cyclic)
    s(1, n_splines=7) +                 # dayofweek
    s(2, n_splines=10, basis="cp") +    # dayofyear (cyclic)
    f(3) +                              # is_holiday (categorical)
    s(4, n_splines=10) +                # load_24
    s(5, n_splines=10) +                # load_168
    s(6, n_splines=10) +                # avg_temp_K
    s(7, n_splines=10) +                # temp_s95
    s(8, n_splines=10) +                # temp_s99
    te(0, 6, n_splines=[8, 8])          # hour x temp interaction
)

print("Fitting GAM with gridsearch...")
gam.gridsearch(X_train, y_train, lam=np.logspace(-3, 3, 11))
print(f"Done — AIC: {gam.statistics_['AIC']:.1f}")

## 8. XGBoost Residual Stacking
No separate validation pass — XGB is fit on training residuals then
evaluated directly on the test set with sliding-window online updates.

In [ ]:
df_train_f = df_train.copy()
df_train_f[FEATURES] = df_train_f[FEATURES].astype(float)

# GAM predictions on training data
gam_pred_train = gam.predict(df_train_f[FEATURES].to_numpy(dtype=float)).reshape(-1)
y_train_actual = df_train_f[TARGET].to_numpy(dtype=float)
residuals_train = y_train_actual - gam_pred_train

# Fit XGBoost on training residuals
X_train_xgb = df_train_f[FEATURES].copy()
X_train_xgb["gam_pred"] = gam_pred_train

gs = GridSearchCV(
    estimator=xgb.XGBRegressor(objective="reg:squarederror", random_state=42),
    param_grid={"n_estimators": [100, 200], "max_depth": [3, 4]},
    cv=3, scoring="neg_mean_squared_error"
)
gs.fit(X_train_xgb, residuals_train)
xgb_model = gs.best_estimator_
print(f"Best XGB params: {gs.best_params_}")

# Save initial state for test loop
xgb_state = {
    "xgb_model":   xgb_model,
    "lambda_val":  1.0,
    "X_history":   X_train_xgb.copy(),
    "res_history": list(residuals_train),
}

## 9. Test Set Evaluation (Jul – Sep 2022) — Primary Reported Results

In [ ]:
df_test_f = df_test.copy()
df_test_f[FEATURES] = df_test_f[FEATURES].astype(float)

gam_pred_test = gam.predict(df_test_f[FEATURES].to_numpy(dtype=float)).reshape(-1)
y_test_actual = df_test_f[TARGET].to_numpy(dtype=float)

# Resume from training state
xgb_model           = xgb_state["xgb_model"]
lambda_val          = xgb_state["lambda_val"]
current_X_history   = xgb_state["X_history"].copy()
current_res_history = list(xgb_state["res_history"])

X_test_xgb_all = df_test_f[FEATURES].copy()
X_test_xgb_all["gam_pred"] = gam_pred_test

updated_preds = []
batch_size  = 24
window      = 2000
tune_window = 168

for i in tqdm(range(0, len(X_test_xgb_all), batch_size), desc="Test batches"):
    X_batch        = X_test_xgb_all.iloc[i : i + batch_size]
    y_batch_actual = y_test_actual[i : i + batch_size]
    gam_batch_pred = gam_pred_test[i : i + batch_size]

    res_pred = xgb_model.predict(X_batch)
    updated_preds.extend(gam_batch_pred + (lambda_val * res_pred))

    observed_residuals = y_batch_actual - gam_batch_pred
    current_res_history.extend(observed_residuals.tolist())
    current_X_history   = pd.concat([current_X_history, X_batch]).iloc[-window:]
    current_res_history = current_res_history[-window:]

    X_train_window   = current_X_history.iloc[:-tune_window]
    res_train_window = current_res_history[:-tune_window]
    X_tune_window    = current_X_history.iloc[-tune_window:]
    res_tune_window  = np.array(current_res_history[-tune_window:], dtype=float)

    xgb_model.fit(X_train_window, res_train_window)
    xgb_preds_tune = xgb_model.predict(X_tune_window)

    def objective_function(l):
        return np.mean((res_tune_window - (l * xgb_preds_tune)) ** 2)

    opt_result = minimize_scalar(objective_function, bounds=(0.0, 1.0), method="bounded")
    lambda_val = opt_result.x

test_preds = np.array(updated_preds, dtype=float)
y_true = y_test_actual[:len(test_preds)]

rmse_test  = np.sqrt(np.mean((y_true - test_preds) ** 2))
wmape_test = np.sum(np.abs(y_true - test_preds)) / np.sum(y_true) * 100
mape_test  = np.mean(np.abs((y_true - test_preds) / (y_true + 1e-5))) * 100

print(f"\n{'TEST SET RESULTS (Jul–Sep 2022) — ERCO':=^50}")
print(f"  RMSE:  {rmse_test:>10.0f} MWh")
print(f"  WMAPE: {wmape_test:>10.2f} %")
print(f"  MAPE:  {mape_test:>10.2f} %")
print(f"  Train period: {df_train['datetime_utc'].min().date()} → {df_train['datetime_utc'].max().date()}")
print(f"  Test period:  {df_test['datetime_utc'].min().date()} → {df_test['datetime_utc'].max().date()}")

## 10. Evaluation Plots

In [ ]:
test_dates = df_test_f["datetime_utc"].values[:len(test_preds)]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Panel 1: Forecast vs Actual — full test period
axes[0].plot(test_dates, y_true, label="Actual", linewidth=1)
axes[0].plot(test_dates, test_preds, label="GAM+XGB", linewidth=1, linestyle="--")
axes[0].set_title("ERCO Load Forecast vs Actual — Test Set (Jul–Sep 2022)")
axes[0].set_ylabel("MWh")
axes[0].legend()

# Panel 2: Residuals
residuals = y_true - test_preds
axes[1].plot(test_dates, residuals, linewidth=0.5, color="firebrick", alpha=0.7)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title(f"Residuals — WMAPE: {wmape_test:.2f}%  |  RMSE: {rmse_test:.0f} MWh")
axes[1].set_ylabel("Residual (MWh)")

plt.tight_layout()
plt.savefig(os.path.expanduser("~/ERCO_forecast_eval.png"), dpi=150)
plt.show()

In [ ]:
# Save predictions for paper figures / further analysis
df_out = pd.DataFrame({
    "datetime_utc": test_dates,
    "actual_mwh":   y_true,
    "predicted_mwh": test_preds,
    "residual_mwh":  residuals,
})
out_path = os.path.expanduser("~/ERCO_wy2022_test_predictions.csv")
df_out.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(df_out):,} rows)")